# ДЗ.7 Векторизация вычислений

Использовать numpy.array вместо list (например, для хранения информации о всех фотонах в событии).
Модифицировать методы для генерации, пропагации фотонов и вычисления числа фотоэлектронов так, чтобы они работали с массивами фотонов (numpy.array). Оценить выигрыш во времени от использования numpy по сравнению с итерацией по отдельным фотонам.

In [32]:
import numpy as np
import matplotlib.pyplot as plt
import time

class Source:
    def __init__(self, Y: int):
        self.Y = Y

    def photon_directions(self, E: float):
       
        N = np.random.poisson(self.Y * E)

        # Генерируем направления сразу для всех фотонов
        phi = np.random.uniform(0, 2*np.pi, N)
        cosTheta = np.random.uniform(-1, 1, N)
        sinTheta = np.sqrt(1 - cosTheta**2)

        x = sinTheta * np.cos(phi)
        y = sinTheta * np.sin(phi)
        z = cosTheta

        return np.array([x, y, z]).T


source = Source(Y=10000)

directions = source.photon_directions(E=1.0)

print(directions.shape)
print(type(directions))

(9974, 3)
<class 'numpy.ndarray'>


In [33]:
import numpy as np

class Detector:
    def __init__(self, R: float, N: int, r: float, a: float, L: float):
     self.R = R
     self.N = N
     self.r = r
     self.a = a
     self.L = L

    def dots_on_parallels(self):

       
        n_theta = 5
        n_phi = 8

        cos_theta = 1 - 2*(np.arange(n_theta) + 0.5)/n_theta
        theta = np.arccos(cos_theta)
        phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)

        dots = []

        for i in range(n_theta):
            for j in range(n_phi):
                x = self.R * np.sin(theta[i]) * np.cos(phi[j])
                y = self.R * np.sin(theta[i]) * np.sin(phi[j])
                z = self.R * np.cos(theta[i])

                dots.append([x, y, z])

        self.photocathode_centers = np.array(dots)

    def is_on_photocathode(self, coordinats):
        

        lengths = np.linalg.norm(self.photocathode_centers - coordinats, axis=1)

        return np.any(lengths <= self.r)

    
    def intersection(self, coordinats, direction):
        projection = np.dot(coordinats, direction)

        distance = -projection + np.sqrt(projection**2 + self.R**2 - np.dot(coordinats, coordinats)) 
        

        point = coordinats + distance * direction

        return self.is_on_photocathode(point)

    def photoelectrons(self):

        return np.random.poisson(self.a)


    def photon_distance(self, coordinats, direction):

        projection = np.dot(coordinats, direction)

        distance = -projection + np.sqrt(
            projection**2 + self.R**2 - np.dot(coordinats, coordinats)
        )

        return distance

    def is_absorbed(self, distance):

        probability = 1 - np.exp(-distance / self.L)

        return np.random.random() < probability

    
    def photons_reach_photocathodes(self, coordinats, directions): # находим точки пересечения всех траекторий с поверхностью детектора 

        projection = np.dot(directions, coordinats)

        distance = -projection + np.sqrt(projection**2 + self.R**2 - np.dot(coordinats, coordinats))

        points = coordinats + distance[:, None] * directions

        # Проверяем попадание фотонов на фотокатоды
        products = np.dot(points, self.photocathode_centers.T)

        # если массив in_ph возвращает True то хотя бы один фотон попал на один фотокатод
        in_ph = np.any(
            products >= self.R**2 - self.r**2 / 2,
            axis=1
        )

        # Учитываем затухание света
        probability = np.exp(-distance / self.L)

        # Проверяем, какие фотоны не поглотились
        survived = np.random.random(len(directions)) < probability

        # условие что фотон должен попал на фотокатод и не поглотился
        return in_ph & survived
#новый метод
    def photoel_array(self, n):
        return np.random.poisson(self.a, size=n)  # генерируем числа фотоэлектронов сразу для n фотонов по распределению Пуассона со средним a

 #это как в 6 дз, здесь добавила для сравнения
    def photon_reaches_photocathode(self, coordinats, direction):
        # Проверяем, достиг ли один фотон фотокатода с учетом затухания

        if not self.intersection(coordinats, direction):
            return False

        distance = self.photon_distance(coordinats, direction)

        return not self.is_absorbed(distance)


detector = Detector(R=2.0, N=40, r=0.1, a=0.5, L=8.0)
detector.dots_on_parallels()

In [34]:
# Моделируем событие с энергией 1 МэВ в центре детектора с помощью np.array

coordinats = np.array([0, 0, 0])
directions = source.photon_directions(E=1.0)

result = detector.photons_reach_photocathodes(coordinats, directions)

n = np.sum(result)
Q = np.sum(detector.photoel_array(n))

print("Число фотонов, достигших фотокатодов:", n)
print("Полный собранный заряд:", Q)


Число фотонов, достигших фотокатодов: 192
Полный собранный заряд: 101


Теперь сравним время обработки фотонов с помощью массивов и по одному как было раньше
(photon_reaches_photocathode - старый метод для одного фотона, а photons_reach_photocathodes - новый метод для массива фотонов)

In [35]:
import time


# Старый способ
start = time.perf_counter()

Q_old = 0

for direction in directions:
    if detector.photon_reaches_photocathode(coordinats, direction):
        Q_old += detector.photoelectrons()

time_old = time.perf_counter() - start

# Новый способ 
start = time.perf_counter()

result = detector.photons_reach_photocathodes(coordinats, directions)
n = np.sum(result)
Q_new = np.sum(detector.photoel_array(n))

time_new = time.perf_counter() - start

print("Время (старый способ):", time_old)
print("Время (новый способ):", time_new)
print("Ускорение в", round(time_old / time_new, 2), "раз")


Время (старый способ): 0.2751259000506252
Время (новый способ): 0.002288299845531583
Ускорение в 120.23 раз
